# SDXL LoRA trainer (advanced)

One cell per stage, so you can check and redo each one. The override dicts in
Configure take any sd-scripts setting.

This notebook asks Colab for an L4 GPU, which needs a [paid Colab plan](https://colab.research.google.com/signup). To use another GPU, open Runtime > Change runtime type. A T4 is slower.

*Built by `scripts/build_notebooks.py` in the repo. Your changes stay in your copy.*

In [ ]:
#@title Setup (installs the trainer in 2 to 4 minutes)
import os, subprocess, sys

ROOT = "/content/kohya-trainer"
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--depth=1", "https://github.com/Linaqruf/kohya-trainer", ROOT], check=True)
    # Fetches the pinned commit, even after upstream moves on.
    subprocess.run(["git", "-C", ROOT, "submodule", "update", "--init", "--depth=1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ROOT], check=True)
# Python loads an editable install only at start, so add src for this session.
sys.path.insert(0, os.path.join(ROOT, "src"))

from kohya_trainer import environment
environment.install()
loaded = environment.load_tokens()
print("runtime:", environment.runtime_report())
print("tokens loaded from Colab Secrets:", ", ".join(loaded) or "none (add HF_TOKEN for gated or private Hugging Face repos, WANDB_API_KEY for log_to_wandb, CIVITAI_TOKEN for Civitai downloads that need a login)")

In [ ]:
#@title Project
project_name = "my-lora" #@param {type:"string"}
workspace_root = "" #@param {type:"string", placeholder:"empty uses Drive if mounted, else /content/lora"}
mount_google_drive = True #@param {type:"boolean"}
#@markdown After a disconnect, tick this to continue from the last save. Keep the model and network fields the same.
resume_from_state = False #@param {type:"boolean"}

from pathlib import Path
from kohya_trainer import environment, train
train.check_project_name(project_name)
if mount_google_drive:
    environment.mount_drive()
workspace = (Path(workspace_root) if workspace_root else environment.default_workspace()) / project_name
workspace.mkdir(parents=True, exist_ok=True)
print("workspace:", workspace)

In [ ]:
#@title Base model
base_model = "animagine-xl-4.0-zero" #@param ["animagine-xl-4.0-zero", "animagine-xl-4.0-opt", "sdxl-base-1.0"]
custom_base = "" #@param {type:"string", placeholder:"a Hugging Face, Civitai or direct .safetensors link"}
#@markdown Tick this for a v-pred `custom_base`, such as a NoobAI v-pred merge. It sets `v_parameterization`, `zero_terminal_snr`, `scale_v_pred_loss_like_noise_pred` and `debiased_estimation_loss` to True, and `sample_sampler` to "euler". It does nothing without a custom base.
custom_base_is_vpred = False #@param {type:"boolean"}

from kohya_trainer import models, registry
route = registry.get("sdxl")
base_spec, checkpoint_path, vae_path, text_encoder_path = models.prepare_base(
    route, None if custom_base else base_model, custom=custom_base)
print("checkpoint:", checkpoint_path)
print("vae:", vae_path)
print("text encoder:", text_encoder_path)

In [ ]:
#@title Dataset
#@markdown A folder, a .zip, a link to a .zip, or a Hugging Face dataset repo of image files or one .zip (user/name). A subfolder named `10_mychar` trains with 10 repeats. A caption file is a .txt with the same name as its image.
dataset_source = "/content/drive/MyDrive/dataset.zip" #@param {type:"string"}

from kohya_trainer import dataset, environment
dataset_dir = dataset.prepare_dataset(dataset_source, environment.scratch_dir(project_name))
print("dataset dir:", dataset_dir)
print(dataset.describe(dataset_dir))

In [ ]:
#@title Captions (tagging and the trigger word)
#@markdown If ticked, the tagger replaces caption files after a copy to `captions-backup` in the workspace. To add to them, tick `append_tags`. To keep them, untick this.
run_tagger = True #@param {type:"boolean"}
tagger = "SmilingWolf/wd-eva02-large-tagger-v3" #@param ["SmilingWolf/wd-eva02-large-tagger-v3", "SmilingWolf/wd-vit-large-tagger-v3"] {allow-input: true}
general_threshold = 0.35 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
character_threshold = 0.35 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
undesired_tags = "" #@param {type:"string", placeholder:"tags the tagger leaves out, such as watermark, signature"}
append_tags = False #@param {type:"boolean"}
character_tags_first = False #@param {type:"boolean"}
#@markdown Start each caption with its rating tag (general, sensitive, questionable or explicit).
use_rating_tags = False #@param {type:"boolean"}
#@markdown Removed from every caption file, with or without the tagger. `long_hair` matches `long hair`.
remove_tags = "" #@param {type:"string", placeholder:"such as simple background, white background"}
trigger_word = "" #@param {type:"string", placeholder:"a unique word such as mychar, added first to every caption (empty keeps your tags)"}
#@markdown After the tagger runs, this removes hair and eye tags found in most captions, so the trigger word learns the look. It works only with a trigger word, and not with `append_tags`.
prune_appearance_tags = True #@param {type:"boolean"}

from kohya_trainer import dataset, tagging
# If you change this, set the same "caption_extension" in dataset_overrides in Configure.
caption_extension = ".txt"
if run_tagger:
    tagging.tag_images(
        dataset_dir,
        repo_id=tagger,
        general_threshold=general_threshold,
        character_threshold=character_threshold,
        undesired_tags=undesired_tags,
        append_tags=append_tags,
        character_tags_first=character_tags_first,
        use_rating_tags=use_rating_tags,
        caption_extension=caption_extension,
        backup_dir=workspace / "captions-backup",
    )
    if prune_appearance_tags and trigger_word.strip() and not append_tags:
        tagging.prune_into_trigger(dataset_dir, caption_extension=caption_extension)
if remove_tags:
    tagging.remove_tags(dataset_dir, remove_tags, caption_extension=caption_extension)
# Put unwanted common tags in undesired_tags or remove_tags, then run this cell again.
print(dataset.tag_report(dataset_dir, caption_extension))

In [ ]:
#@title Configure
#@markdown **Network**. The defaults set alpha to half of dim, so change them together.
#@markdown LoHa and LoKr ignore these two fields, run about twice as slow and need more GPU memory. To size them, add `"network_dim"` and `"network_alpha"` (LoHa, 16 and 8 by default) or `"network_args": ["factor=8"]` (LoKr) to `config_overrides`. On a T4, if it runs out of memory, set `train_batch_size` to 1.
#@markdown If you pick lokr, set `learning_rate` to about 1e-3. At 3e-4, full-size LoKr learns a character only weakly.
network_type = "lora" #@param ["lora", "locon", "loha", "lokr"]
network_dim = 32 #@param {type:"integer"}
network_alpha = 16 #@param {type:"number"}
#@markdown Only LoCon uses the conv layers.
conv_dim = 8 #@param {type:"integer"}
conv_alpha = 4 #@param {type:"number"}
network_weights = "" #@param {type:"string", placeholder:"a LoRA file or link to keep training (empty starts fresh)"}
#@markdown ---
#@markdown **Training**
max_train_epochs = 10 #@param {type:"integer"}
#@markdown 0 uses the epoch count.
max_train_steps = 0 #@param {type:"integer"}
train_batch_size = 2 #@param {type:"integer"}
#@markdown Works like a bigger batch, with no extra GPU memory.
gradient_accumulation_steps = 1 #@param {type:"integer"}
resolution = 1024 #@param {type:"integer"}
#@markdown 0 picks the repeats for about 400 images per epoch.
num_repeats = 0 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}
#@markdown ---
#@markdown **Learning rate**
learning_rate = 0.0003 #@param {type:"number"}
#@markdown Pick one or type another sd-scripts name. Schedule-free optimizers ignore the scheduler and warmup. With Prodigy, cosine or constant_with_warmup are the usual picks.
lr_scheduler = "cosine" #@param ["constant", "constant_with_warmup", "cosine", "cosine_with_restarts", "linear"] {allow-input: true}
#@markdown Only cosine_with_restarts uses the cycles.
lr_scheduler_num_cycles = 3 #@param {type:"integer"}
#@markdown A share of all steps (0.05 is 5%) or a step count.
lr_warmup_steps = 0.05 #@param {type:"number"}
#@markdown Min-SNR gamma: 5 is usual, 0 turns it off. Most current recipes leave `noise_offset` at 0. A v-pred base drops both.
min_snr_gamma = 5 #@param {type:"number"}
noise_offset = 0 #@param {type:"number"}
#@markdown ---
#@markdown **Optimizer**. To leave out the optimizer's recommended settings and keep your own learning rate, untick `apply_optimizer_defaults`.
#@markdown Type any sd-scripts optimizer, or one from an installed package as package.Name, such as pytorch_optimizer.CAME. CAME usually runs at 1e-4 to 2e-4. The optimizer defaults for Prodigy and ProdigyScheduleFree set your learning rate to 1.0.
optimizer = "AdamW8bit" #@param ["AdamW8bit", "AdamW", "Prodigy", "ProdigyScheduleFree", "CAME", "AdaFactor"] {allow-input: true}
apply_optimizer_defaults = True #@param {type:"boolean"}
optimizer_args = "" #@param {type:"string", placeholder:"key=value settings, such as weight_decay=0.05 (yours replace the defaults)"}
#@markdown ---
#@markdown **Saving and output**
#@markdown 0 saves only the final LoRA file.
save_every_n_epochs = 1 #@param {type:"integer"}
#@markdown Keep only the newest N epoch files. 0 keeps them all.
save_last_n_epochs = 0 #@param {type:"integer"}
sample_prompt = "" #@param {type:"string", placeholder:"a prompt, or a .txt file with one per line (empty for the built-in one)"}
sample_negative_prompt = "nsfw, lowres, bad anatomy, bad hands, text, error, missing finger, extra digits, fewer digits, cropped, worst quality, low quality, low score, bad score, average score, signature, watermark, username, blurry" #@param {type:"string"}
sample_every_n_epochs = 1 #@param {type:"integer"}
upload_to_hf = "" #@param {type:"string", placeholder:"username/repo for each saved epoch (needs a write HF_TOKEN)"}
make_repo_public = False #@param {type:"boolean"}
#@markdown Needs the `WANDB_API_KEY` secret.
log_to_wandb = False #@param {type:"boolean"}

# The form's training fields. 0 turns a number off.
form_values = {
    "learning_rate": learning_rate,
    "max_train_epochs": max_train_epochs,
    "max_train_steps": max_train_steps,
    "gradient_accumulation_steps": gradient_accumulation_steps,
    "seed": seed,
    "save_every_n_epochs": save_every_n_epochs,
    "save_last_n_epochs": save_last_n_epochs,
    "lr_scheduler": lr_scheduler,
    "lr_scheduler_num_cycles": lr_scheduler_num_cycles,
    "lr_warmup_steps": lr_warmup_steps,
    "sample_every_n_epochs": sample_every_n_epochs,
    "min_snr_gamma": min_snr_gamma,
    "noise_offset": noise_offset,
}
# Other sd-scripts settings go in config_overrides, dataset settings in dataset_overrides. Both win over the form and the preset.
config_overrides = {}
dataset_overrides = {}

from kohya_trainer import train
plan = train.build_plan(
    route,
    base_spec,
    checkpoint_path=checkpoint_path,
    vae_path=vae_path,
    text_encoder_path=text_encoder_path,
    dataset_dir=dataset_dir,
    workspace=workspace,
    project_name=project_name,
    trigger_word=trigger_word,
    network_type=network_type,
    network_dim=network_dim,
    network_alpha=network_alpha,
    conv_dim=conv_dim,
    conv_alpha=conv_alpha,
    network_weights=network_weights,
    optimizer=optimizer,
    optimizer_args=optimizer_args,
    optimizer_recommended=apply_optimizer_defaults,
    resume=resume_from_state,
    upload_repo=upload_to_hf,
    upload_public=make_repo_public,
    log_to_wandb=log_to_wandb,
    custom_base_is_vpred=custom_base_is_vpred,
    resolution=resolution,
    train_batch_size=train_batch_size,
    num_repeats=num_repeats or None,
    sample_prompt=sample_prompt or None,
    sample_negative_prompt=sample_negative_prompt,
    form_values=form_values,
    config_overrides=config_overrides,
    dataset_overrides=dataset_overrides,
)
print("training config:", plan.config_file)
print("dataset config:", plan.dataset_config_file)
print("launch:", " ".join(plan.command))

In [ ]:
#@title Train
from kohya_trainer import train
train.execute(plan)

### Notes

- After a change, run that cell and the cells after it. Nothing downloads twice.
- With `save_every_n_epochs = 1` (the preset default), every finished epoch is already saved. Keep the
  workspace on Drive, the default when mounted. Colab deletes /content when a session ends.
- A session keeps running and using compute units after the page disconnects. If the page comes back on a
  new session, stop the old one in Runtime > Manage sessions. When you are done, choose Runtime >
  Disconnect and delete runtime.
- If every image shows one outfit, a prompt for that outfit can bring the subject in without the trigger word. If that is a problem, lower the LoRA weight.
- Caption shuffle is off, because the text-encoder outputs are cached next to the images. To shuffle, add
  `"cache_text_encoder_outputs": None, "cache_text_encoder_outputs_to_disk": None` to `config_overrides`
  and set `dataset_overrides = {"shuffle_caption": True, "keep_tokens": 1}`. This uses more GPU memory.
- If a caption, the trigger word, `caption_prefix` or the text encoder changes, Train rebuilds that cache.
  Run one training at a time on a folder, because a second run replaces the first run's cache.
- `config_overrides` takes any sd-scripts argument (`python external/sd-scripts/sdxl_train_network.py --help`).
  `dataset_overrides` takes the arguments of `kohya_trainer.dataset.build_dataset_config`: shuffle_caption,
  keep_tokens, flip_aug, caption_extension, class_tokens, caption_prefix, resolution, batch_size,
  num_repeats and the bucket settings. A numbered folder's own number and name (such as 10_mychar) win over
  num_repeats and class_tokens there. Other dataset settings, such as caption_dropout_rate or color_aug,
  move to `config_overrides`. Any other key is refused.
- Settings people often add in Configure's code:
  - config_overrides: `"caption_dropout_rate": 0.1` trains one image in ten without its caption. It needs text-encoder caching turned off, as for caption shuffle above.
  - config_overrides: `"save_every_n_steps": 500, "save_last_n_steps": 1500` also saves every 500 steps and keeps only the step files from the last 1500 steps.
  - config_overrides: `"min_timestep": 0, "max_timestep": 700` trains only the lower noise levels (of 1000), for detail over composition.
  - config_overrides: `"network_dropout": 0.1` can help against overfitting.
  - dataset_overrides: `{"flip_aug": True}` also trains on mirrored images. Leave it off for an asymmetric subject.
  - config_overrides: `"multires_noise_iterations": 6, "multires_noise_discount": 0.3` widens the range of contrast and detail.
  - config_overrides: `"lr_decay_steps": 0.1` with `lr_scheduler = "warmup_stable_decay"` (typed in) holds the rate after warmup and lowers it over the last 10% of the steps.
- The presets in `src/kohya_trainer/presets/` give the form its defaults and fill in every setting the
  form does not show. The form wins over the preset, and the two dicts win over both. To change a value
  the form shows, change the field.